# DMR Analysis — Prototyping Notebook

Prototype for **Differentially Methylated Region (DMR)** detection from **ONT native methylation** BAMs.  
Data was produced by **dorado duplex** with `4mC_5mC,6mA` modification calling.

**Tool chain:**
1. `modkit pileup` — extract per-CpG methylation from MM/ML tags → bedMethyl format
2. `bedmethyl_to_dss.py` — reformat bedMethyl → DSS-compatible TSV (`chr pos N X`)
3. `DSS` (R) — `DMLtest` + `callDMR`

**Current samples:** r0066 *(one sample; DMR calling enabled once a second sample is available)*

**Final Snakemake workflow:** `workflows/dmr_analysis/Snakefile`

In [ ]:
import subprocess
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ── Paths ─────────────────────────────────────────────────────────────────────
WDIR        = Path('/home/daffa/Work/2026/02-JSPP67')
DMR_BAM_DIR = Path('/home/daffa/hdd/2026/02-JSPP67_hdd/DMR/trim_bam')
REF         = WDIR / 'data/reference/asm_BTx623.fna'
RESULTS_DIR = WDIR / 'results/dmr_analysis'
SCRIPTS_DIR = WDIR / 'workflows/dmr_analysis/scripts'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES = ['r0066']
BAMS    = {s: DMR_BAM_DIR / f'{s}.bam' for s in SAMPLES}

print('Reference :', REF, '— exists:', REF.exists())
print('BAMs      :')
for s, b in BAMS.items():
    print(f'  {s}: {b}  exists={b.exists()}')

## Step 1 — Inspect the ONT methylation BAM

Confirm `@PG` tags show dorado with modification calling, and that MM/ML tags are present.

In [ ]:
# Inspect BAM header @PG tags
sample = 'r0066'
bam    = BAMS[sample]

header = subprocess.run(
    ['samtools', 'view', '-H', str(bam)],
    capture_output=True, text=True
).stdout

print('=== @PG tags ===')
for line in header.splitlines():
    if line.startswith('@PG') or line.startswith('@RG'):
        print(line[:200])

# Check first read for MM/ML tags
print('\n=== First read (MM/ML check) ===')
first_read = subprocess.run(
    ['samtools', 'view', str(bam)],
    capture_output=True, text=True
).stdout.splitlines()[:1]
if first_read:
    fields = first_read[0].split('\t')
    mm_ml = [f for f in fields if f.startswith('MM:') or f.startswith('ML:')]
    print('MM/ML tags present:', bool(mm_ml))
    for tag in mm_ml:
        print(' ', tag[:120])

## Step 2 — modkit pileup (per-CpG methylation)

`modkit pileup --preset traditional` extracts **CpG 5mC** calls from MM/ML tags.

Output: **bedMethyl** format (bed9+) with columns including `N_valid_cov` and `N_mod`.

In [ ]:
# ── modkit pileup ─────────────────────────────────────────────────────────────
proto_sample = 'r0066'
proto_bam    = BAMS[proto_sample]
proto_dir    = RESULTS_DIR / proto_sample
proto_dir.mkdir(parents=True, exist_ok=True)

bedmethyl_out = proto_dir / f'{proto_sample}_CpG.bedMethyl'

modkit_cmd = [
    'modkit', 'pileup',
    str(proto_bam), str(bedmethyl_out),
    '--ref', str(REF),
    '--preset', 'traditional',
    '--ignore', 'h',          # ignore 5hmC
    '--threads', '8',
]

print('Command:\n  ' + ' '.join(modkit_cmd))
print('\nRunning modkit pileup … (may take several minutes)')
result = subprocess.run(modkit_cmd, capture_output=True, text=True)
print(result.stderr[-2000:] if result.stderr else '')
if result.returncode == 0:
    print('Done. Output:', bedmethyl_out)
else:
    print('ERROR:', result.returncode)
    print(result.stdout)

## Step 3 — Explore bedMethyl output

In [ ]:
# Load bedMethyl
# columns: chrom start end mod_code score strand start2 end2 color
#          N_valid_cov pct_mod N_mod N_canonical N_other_mod N_del N_fail N_diff N_nocall
cols = [
    'chrom', 'start', 'end', 'mod_code', 'score', 'strand',
    'start2', 'end2', 'color',
    'N_valid_cov', 'pct_mod', 'N_mod', 'N_canonical',
    'N_other_mod', 'N_del', 'N_fail', 'N_diff', 'N_nocall'
]

df = pd.read_csv(bedmethyl_out, sep='\t', names=cols, comment='#')
df = df[df['N_valid_cov'] > 0].copy()

print(f'Total CpG sites with coverage: {len(df):,}')
print(f'Median coverage : {df.N_valid_cov.median():.1f}x')
print(f'Mean methylation: {df.pct_mod.mean():.1f}%')
print()
print('Per-chromosome CpG count and mean methylation:')
print(
    df.groupby('chrom').agg(
        n_sites=('start', 'count'),
        mean_cov=('N_valid_cov', 'mean'),
        mean_meth=('pct_mod', 'mean')
    ).round(1).head(12)
)

In [ ]:
# Plot methylation distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['N_valid_cov'].clip(upper=100), bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Coverage (capped at 100)')
axes[0].set_ylabel('Number of CpG sites')
axes[0].set_title(f'{proto_sample} — Coverage distribution')

axes[1].hist(df['pct_mod'], bins=50, color='darkorange', edgecolor='white')
axes[1].set_xlabel('Methylation (%)')
axes[1].set_ylabel('Number of CpG sites')
axes[1].set_title(f'{proto_sample} — Methylation distribution')

plt.tight_layout()
plt.savefig(proto_dir / f'{proto_sample}_methylation_distribution.png', dpi=150)
plt.show()

## Step 4 — Convert bedMethyl → DSS format

In [ ]:
dss_out = proto_dir / f'{proto_sample}_CpG.dss.tsv'

result = subprocess.run(
    ['python', str(SCRIPTS_DIR / 'bedmethyl_to_dss.py'), str(bedmethyl_out), str(dss_out)],
    capture_output=True, text=True
)
print(result.stderr)

dss_df = pd.read_csv(dss_out, sep='\t')
print(f'DSS input rows: {len(dss_df):,}')
print(dss_df.head())

## Step 5 — DMR calling with DSS (stub — requires ≥ 2 samples)

The R code below shows what `dss_dmr.R` will execute.  
To activate: add a second sample to `dmr_samples` and add a comparison to `dmr_comparisons` in `configs/config.yaml`.

In [ ]:
# This cell is a stub — will run once a second sample is available.
# Equivalent to the Snakemake call_dmr rule.

SAMPLE_A = 'r0066'          # control / reference methylation
SAMPLE_B = 'r0067'          # new sample to compare (placeholder)
COMPARISON_NAME = f'{SAMPLE_A}_vs_{SAMPLE_B}'

dss_a = RESULTS_DIR / SAMPLE_A / f'{SAMPLE_A}_CpG.dss.tsv'
dss_b = RESULTS_DIR / SAMPLE_B / f'{SAMPLE_B}_CpG.dss.tsv'   # does not exist yet

comp_out = RESULTS_DIR / COMPARISON_NAME / 'dmr_results.tsv'
comp_out.parent.mkdir(parents=True, exist_ok=True)

dss_cmd = [
    'Rscript', str(SCRIPTS_DIR / 'dss_dmr.R'),
    '--sample_a', str(dss_a),
    '--sample_b', str(dss_b),
    '--comparison', COMPARISON_NAME,
    '--delta', '0.1',
    '--p_threshold', '0.05',
    '--min_cpg', '3',
    '--min_len', '50',
    '--output', str(comp_out),
]

print('DSS command (ready to run once sample_b exists):')
print('  ' + ' '.join(dss_cmd))
print(f'\nSample B DSS file exists: {dss_b.exists()}')
if dss_b.exists():
    result = subprocess.run(dss_cmd, capture_output=True, text=True)
    print(result.stdout[-1000:])
    if result.returncode == 0 and comp_out.exists():
        dmr_df = pd.read_csv(comp_out, sep='\t')
        print(f'\nFound {len(dmr_df)} DMRs:')
        print(dmr_df.head())
else:
    print('[STUB] Add sample_b and re-run this cell.')